In [1]:
print("hi")

hi


In [2]:
%pwd

'c:\\Users\\Pratham\\OneDrive\\Desktop\\sezzle-ai-customer-support-agent\\research'

In [3]:
#since I am right now in research folder(-\\research). I have to come back and enter into Data folder

In [4]:
import os 
os.chdir("../")

In [5]:
%pwd

'c:\\Users\\Pratham\\OneDrive\\Desktop\\sezzle-ai-customer-support-agent'

1. DATA EXTRACTION OF THE JOSN FILE.

In [6]:
import json
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter

c:\Users\Pratham\anaconda3\envs\sezzle_bot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
from langchain_community.document_loaders import DirectoryLoader, JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [9]:
import json
from langchain.schema import Document

def load_json_file(file_path):
    """
    Parses a nested JSON file and returns a list of LangChain Document objects.
    Aligns exactly with the structural content layout of the help center data.
    """
    documents = []

    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        # Loop through each parent grouping/category block
        for category_item in data:
            category_title = category_item.get("title", "").strip()
            articles = category_item.get("articles", [])

            # Loop through each individual article block
            for article in articles:
                article_title = article.get("title", "").strip()
                article_url = article.get("url", "").strip()
                breadcrumbs = article.get("breadcrumbs", [])
                content_blocks = article.get("content", [])

                # ---- Extract and Reconstruct Structural Inner Body Text ----
                text_segments = []
                for block in content_blocks:
                    block_type = block.get("type")
                    block_text = block.get("text", "").strip()
                    
                    if not block_text:
                        continue
                        
                    # Handle lists/bullet styling parameters cleanly
                    if block_type == "bullet":
                        text_segments.append(f"• {block_text}")
                    elif block_type == "number":
                        num = block.get("number", "")
                        text_segments.append(f"{num}. {block_text}")
                    else:
                        text_segments.append(block_text)

                # Combine all localized strings into a single cohesive body
                full_content_text = "\n\n".join(text_segments)

                # ---- Minimal Metadata Definition ----
                metadata = {
                    "category": category_title,
                    "title": article_title,
                    "url": article_url,
                    "breadcrumbs": " > ".join(breadcrumbs) if breadcrumbs else ""
                }

                # ---- Rich Page Content Construction ----
                page_content = f"""
Category Context: {category_title}
Article Title: {article_title}
Help Document Source Link: {article_url}
Navigation Path: {" > ".join(breadcrumbs) if breadcrumbs else "N/A"}

Document Body Content:
{full_content_text}
"""

                # Prevent appending empty document frames into vector storage
                if full_content_text.strip():
                    documents.append(
                        Document(
                            metadata=metadata,
                            page_content=page_content.strip()
                        )
                    )

        return documents

    except FileNotFoundError:
        print(f"❌ Error: File not found at {file_path}")
        return []
    except json.JSONDecodeError:
        print(f"❌ Error: Failed to decode JSON formatting inside {file_path}")
        return []
    except Exception as e:
        print(f"❌ Unexpected error occurred during file parsing: {e}")
        return []

In [ ]:
extracted_docs = load_json_file("data/merchant_help_center.json")

print("Total Documents:", len(extracted_docs))
print(extracted_docs[0])

Total Documents: 68
page_content='Category Context: Sezzle Basics Sezzle products and services for you and your shoppers.
Article Title: Commonly asked questions for prospective merchants
Help Document Source Link: https://merchant-help.sezzle.com/hc/en-us/articles/17513899360404-Commonly-asked-questions-for-prospective-merchants
Navigation Path: Sezzle Merchant Support > Sezzle Basics > Sezzle Merchant Resources

Document Body Content:
If you're still unsure about signing up for Sezzle, here’s a quick one-pager of our most common inquiries from new merchants.

How do I sign up for Sezzle as a merchant?

Learn more about Sezzle for Merchants and start the application process at https://sezzle.com/merchants .

How do I integrate Sezzle into my website?

Once your account has been approved, you’ll be sent an email with instructions to log in and get set up. If you lose that email or if you get stuck during the setup process, you can always log in to your account and select "Setup Checkli

In [14]:
extracted_docs = load_json_file("data/partner_help_center.json")

print("Total Documents:", len(extracted_docs))
print(extracted_docs[0])

Total Documents: 10
page_content='Category Context: Getting Started Get started as a Sezzle Partner — from creating your account to completing onboarding and sending your first referrals.
Article Title: Issues Signing Up
Help Document Source Link: https://partnersupport-sezzle.zendesk.com/hc/en-us/articles/40302855673108-Issues-Signing-Up
Navigation Path: Partner Support > Getting Started > Sign-Up & Onboarding

Document Body Content:
If you are unable to log in or are having issues signing up for a partner account, it may be because you already have a merchant or shopper account with Sezzle. If so, we recommend opening a private or incognito browser window to complete the partner sign-up process.' metadata={'category': 'Getting Started Get started as a Sezzle Partner — from creating your account to completing onboarding and sending your first referrals.', 'title': 'Issues Signing Up', 'url': 'https://partnersupport-sezzle.zendesk.com/hc/en-us/articles/40302855673108-Issues-Signing-Up'

In [15]:
extracted_docs = load_json_file("data/shopper_help_center.json")

print("Total Documents:", len(extracted_docs))
print(extracted_docs[0])

Total Documents: 112
page_content='Category Context: Account Settings, Security, & Subscriptions Login troubleshooting, managing your personal information and payment methods, and account security best practices.
Article Title: I need help getting logged in.
Help Document Source Link: https://shopper-help.sezzle.com/hc/en-us/articles/360046954031-I-need-help-getting-logged-in
Navigation Path: Sezzle > Account Settings, Security, & Subscriptions > Login Troubleshooting

Document Body Content:
If you have trouble logging into your Sezzle account, you’ve come to the right place! See the instructions and resources below to help you navigate the various factors needed to log in.

I forgot my PIN.

To log in to your Sezzle account, you will need your phone number and the 4-digit PIN you selected when you created your account. If you need to reset your PIN, follow the steps here .

I never set-up a PIN and don't have access to my phone/email.

If you never set up a PIN when logging into Sezzl

In [12]:
extracted_docs

[Document(metadata={'category': 'Sezzle Basics Sezzle products and services for you and your shoppers.', 'title': 'Commonly asked questions for prospective merchants', 'url': 'https://merchant-help.sezzle.com/hc/en-us/articles/17513899360404-Commonly-asked-questions-for-prospective-merchants', 'breadcrumbs': 'Sezzle Merchant Support > Sezzle Basics > Sezzle Merchant Resources'}, page_content='Category Context: Sezzle Basics Sezzle products and services for you and your shoppers.\nArticle Title: Commonly asked questions for prospective merchants\nHelp Document Source Link: https://merchant-help.sezzle.com/hc/en-us/articles/17513899360404-Commonly-asked-questions-for-prospective-merchants\nNavigation Path: Sezzle Merchant Support > Sezzle Basics > Sezzle Merchant Resources\n\nDocument Body Content:\nIf you\'re still unsure about signing up for Sezzle, here’s a quick one-pager of our most common inquiries from new merchants.\n\nHow do I sign up for Sezzle as a merchant?\n\nLearn more abou

In [13]:
import os

# Define the paths to your 3 uploaded JSON files
json_files = {
    "Shopper Help Center": "data/shopper_help_center.json",
    "Merchant Help Center": "data/merchant_help_center.json",
    "Partner Help Center": "data/partner_help_center.json"
}

all_extracted_docs = []

print("🚀 Starting multi-file document extraction...\n")

# Loop through each file and append the documents
for source_name, file_path in json_files.items():
    if not os.path.exists(file_path):
        print(f"⚠️  WARNING: File missing at '{file_path}'. Skipping...")
        continue
        
    # Call the reference loader function
    file_docs = load_json_file(file_path)
    all_extracted_docs.extend(file_docs)
    
    print(f"📁 {source_name}: Extracted {len(file_docs)} documents.")

print("\n📊 Extraction Complete!")
print("====================================")
print("Total Combined Documents:", len(all_extracted_docs))
print("====================================\n")

# Verify by printing the first document block if data was loaded successfully
if all_extracted_docs:
    print("🧪 First Extracted Document Sample:")
    print("-" * 40)
    print(all_extracted_docs[0])
else:
    print("❌ No documents found. Please verify your file names and folder paths.")

🚀 Starting multi-file document extraction...

📁 Shopper Help Center: Extracted 112 documents.
📁 Merchant Help Center: Extracted 68 documents.
📁 Partner Help Center: Extracted 10 documents.

📊 Extraction Complete!
Total Combined Documents: 190

🧪 First Extracted Document Sample:
----------------------------------------
page_content='Category Context: Account Settings, Security, & Subscriptions Login troubleshooting, managing your personal information and payment methods, and account security best practices.
Article Title: I need help getting logged in.
Help Document Source Link: https://shopper-help.sezzle.com/hc/en-us/articles/360046954031-I-need-help-getting-logged-in
Navigation Path: Sezzle > Account Settings, Security, & Subscriptions > Login Troubleshooting

Document Body Content:
If you have trouble logging into your Sezzle account, you’ve come to the right place! See the instructions and resources below to help you navigate the various factors needed to log in.

I forgot my PIN.


In [16]:
all_extracted_docs

[Document(metadata={'category': 'Account Settings, Security, & Subscriptions Login troubleshooting, managing your personal information and payment methods, and account security best practices.', 'title': 'I need help getting logged in.', 'url': 'https://shopper-help.sezzle.com/hc/en-us/articles/360046954031-I-need-help-getting-logged-in', 'breadcrumbs': 'Sezzle > Account Settings, Security, & Subscriptions > Login Troubleshooting'}, page_content="Category Context: Account Settings, Security, & Subscriptions Login troubleshooting, managing your personal information and payment methods, and account security best practices.\nArticle Title: I need help getting logged in.\nHelp Document Source Link: https://shopper-help.sezzle.com/hc/en-us/articles/360046954031-I-need-help-getting-logged-in\nNavigation Path: Sezzle > Account Settings, Security, & Subscriptions > Login Troubleshooting\n\nDocument Body Content:\nIf you have trouble logging into your Sezzle account, you’ve come to the right pl

In [17]:
# Since the page content is not cleaned like \n next page , etc...
import re

def clean_product_text(text):

    # --- FIX WORDS BROKEN BY HYPHEN + NEWLINE ---
    text = re.sub(r'(\w+)-\s*[\r\n]+\s*(\w+)', r'\1\2', text)

    # --- PROTECT PARAGRAPHS ---
    text = text.replace('\n\n', '[[PARA]]')

    # --- REMOVE SINGLE NEWLINES/TABS ---
    text = text.replace('\n', '.').replace('\r', ' ').replace('\t', ' ')

    # --- RESTORE PARAGRAPHS ---
    text = text.replace('[[PARA]]', '\n\n')

    # --- REMOVE EXTRA SPACES ---
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [18]:
def clean_documents(documents):

    for doc in documents:
        doc.page_content = clean_product_text(doc.page_content)

    return documents

In [19]:
# Clean the documents
cleaned_docs = clean_documents(all_extracted_docs)

print("Total Documents:", len(cleaned_docs))
print(cleaned_docs[0]) # checking if above function work or not.

Total Documents: 190
page_content='Category Context: Account Settings, Security, & Subscriptions Login troubleshooting, managing your personal information and payment methods, and account security best practices..Article Title: I need help getting logged in..Help Document Source Link: https://shopper-help.sezzle.com/hc/en-us/articles/360046954031-I-need-help-getting-logged-in.Navigation Path: Sezzle > Account Settings, Security, & Subscriptions > Login Troubleshooting Document Body Content:.If you have trouble logging into your Sezzle account, you’ve come to the right place! See the instructions and resources below to help you navigate the various factors needed to log in. I forgot my PIN. To log in to your Sezzle account, you will need your phone number and the 4-digit PIN you selected when you created your account. If you need to reset your PIN, follow the steps here . I never set-up a PIN and don't have access to my phone/email. If you never set up a PIN when logging into Sezzle and

In [20]:
cleaned_docs

[Document(metadata={'category': 'Account Settings, Security, & Subscriptions Login troubleshooting, managing your personal information and payment methods, and account security best practices.', 'title': 'I need help getting logged in.', 'url': 'https://shopper-help.sezzle.com/hc/en-us/articles/360046954031-I-need-help-getting-logged-in', 'breadcrumbs': 'Sezzle > Account Settings, Security, & Subscriptions > Login Troubleshooting'}, page_content="Category Context: Account Settings, Security, & Subscriptions Login troubleshooting, managing your personal information and payment methods, and account security best practices..Article Title: I need help getting logged in..Help Document Source Link: https://shopper-help.sezzle.com/hc/en-us/articles/360046954031-I-need-help-getting-logged-in.Navigation Path: Sezzle > Account Settings, Security, & Subscriptions > Login Troubleshooting Document Body Content:.If you have trouble logging into your Sezzle account, you’ve come to the right place! Se

2. CHUNKING

In [21]:
# For product catalog we don't require chunking coz already we have one product in one documnent. So we can directly use cleaned_docs for vectorization and embedding.

In [22]:
import json
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter

# ── Step 1: Content blocks → clean text ──────────────────────────────────────

def blocks_to_text(content_blocks: list) -> str:
    """
    Convert JSON content blocks into clean readable text.
    Preserves structure: headings, bullets, numbered steps.
    """
    lines = []
    for block in content_blocks:
        btype = block.get("type", "")
        text  = block.get("text", "").strip()
        if not text:
            continue
        if btype == "heading":
            lines.append(f"\n{text}:")
        elif btype == "bullet":
            lines.append(f"• {text}")
        elif btype == "number":
            lines.append(f"{block.get('number', '')}. {text}")
        else:  # paragraph
            lines.append(text)
    return "\n".join(lines).strip()

In [23]:
blocks_to_text(extracted_docs[0].page_content)

AttributeError: 'str' object has no attribute 'get'

In [26]:
import json
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter

# ── Step 1: Content blocks → clean text ──────────────────────────────────────

def blocks_to_text(content_blocks: list) -> str:
    """
    Convert JSON content blocks into clean readable text.
    Preserves structure: headings, bullets, numbered steps.
    """
    lines = []
    for block in content_blocks:
        btype = block.get("type", "")
        text  = block.get("text", "").strip()
        if not text:
            continue
        if btype == "heading":
            lines.append(f"\n{text}:")
        elif btype == "bullet":
            lines.append(f"• {text}")
        elif btype == "number":
            lines.append(f"{block.get('number', '')}. {text}")
        else:  # paragraph
            lines.append(text)
    return "\n".join(lines).strip()


# ── Step 2: JSON → LangChain Documents ───────────────────────────────────────

def load_sezzle_json(filepath: str, source: str) -> list[Document]:
    """
    Load one Sezzle JSON file.
    Each article becomes one Document with full metadata.
    """
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)

    documents = []
    for section in data:
        section_title = section.get("title", "")[:60]
        for article in section.get("articles", []):
            body = blocks_to_text(article.get("content", []))
            if not body:
                continue

            # Full text = title + body (title helps retrieval)
            full_text = f"{article['title']}\n\n{body}"

            doc = Document(
                page_content=full_text,
                metadata={
                    "title":   article["title"],
                    "url":     article.get("url", ""),
                    "source":  source,
                    "section": section_title,
                }
            )
            documents.append(doc)

    return documents


# ── Step 3: Smart chunking ────────────────────────────────────────────────────

def chunk_documents(documents: list[Document]) -> list[Document]:
    """
    Chunk strategy:
    - Short articles  (<=500 chars) → keep as-is, no split
    - Medium articles (<=1200 chars) → keep as-is
    - Long articles   (>1200 chars)  → split with overlap

    Why 500/1200 thresholds?
    - Embedding model (all-MiniLM-L6-v2) sweet spot is 256-512 tokens
    - ~500 chars ≈ 100 tokens, ~1200 chars ≈ 250 tokens
    - Splitting short articles loses context
    """
    short_docs  = []  # <= 1200 chars — keep whole
    long_docs   = []  # >  1200 chars — need splitting

    for doc in documents:
        if len(doc.page_content) <= 1200:
            short_docs.append(doc)
        else:
            long_docs.append(doc)

    # Split only long articles
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
        separators=["\n\n", "\n", "•", ". ", " "],
    )
    split_chunks = splitter.split_documents(long_docs)

    # Preserve metadata on split chunks
    all_chunks = short_docs + split_chunks
    return all_chunks


# ── Step 4: Run everything ────────────────────────────────────────────────────

shopper_docs  = load_sezzle_json(
    "data/shopper_help_center.json",
    "shopper-help.sezzle.com"
)
merchant_docs = load_sezzle_json(
    "data/merchant_help_center.json",
    "merchant-help.sezzle.com"
)

all_docs   = shopper_docs + merchant_docs
all_chunks = chunk_documents(all_docs)

# ── Step 5: Stats ─────────────────────────────────────────────────────────────

print(f"Shopper articles  : {len(shopper_docs)}")
print(f"Merchant articles : {len(merchant_docs)}")
print(f"Total articles    : {len(all_docs)}")
print(f"Total chunks      : {len(all_chunks)}")
print()

# Size distribution
short = sum(1 for d in all_docs if len(d.page_content) <= 1200)
long  = sum(1 for d in all_docs if len(d.page_content) >  1200)
print(f"Short articles (kept whole) : {short}")
print(f"Long articles  (split)      : {long}")
print()

# Sample chunk
print("=== SAMPLE CHUNK ===")
sample = all_chunks[10]
print(f"Title   : {sample.metadata['title']}")
print(f"Source  : {sample.metadata['source']}")
print(f"Section : {sample.metadata['section']}")
print(f"Length  : {len(sample.page_content)} chars")
print(f"Content preview:\n{sample.page_content[:]}")

Shopper articles  : 112
Merchant articles : 68
Total articles    : 180
Total chunks      : 869

Short articles (kept whole) : 74
Long articles  (split)      : 106

=== SAMPLE CHUNK ===
Title   : How do promotions work with Sezzle?
Source  : shopper-help.sezzle.com
Section : Learn More About Sezzle Products Sezzle basics, premium perk
Length  : 1031 chars
Content preview:
How do promotions work with Sezzle?

Sezzle will occasionally offer promotions with our partnered stores or other financial institutions. Promotions may last for a limited time or be ongoing. The rewards offered for a promotion can be in the form of Sezzle Spend or a discount on a qualifying order at checkout. The following are examples of the types of promotions we offer:
• Sezzle shoppers earn Sezzle Spend when shopping with specific stores.
• A promo code can apply a discount to a qualifying store at checkout.
• Sezzle Spend can be issued after placing an order. This can be used on future purchases with Sezzle. Clic

3. EMBEDDING MODELS

In [27]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)

In [28]:
embeddings

HuggingFaceEmbeddings(model_name='BAAI/bge-base-en-v1.5', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [29]:
vector = embeddings.embed_query("Hello world")
vector

[0.010723961517214775,
 0.05578305199742317,
 0.02708449587225914,
 0.00304088881239295,
 0.030335664749145508,
 0.020480157807469368,
 0.03132810816168785,
 0.041038878262043,
 -0.025208471342921257,
 -0.057271808385849,
 -0.00396144762635231,
 -0.004360385704785585,
 -0.06811613589525223,
 0.019528986886143684,
 0.016956165432929993,
 0.028180859982967377,
 0.03159738704562187,
 0.0007254044758155942,
 0.015515279956161976,
 0.03791613131761551,
 -0.05291657894849777,
 0.009345244616270065,
 0.03269641846418381,
 0.015812644734978676,
 -0.006123300176113844,
 -0.007728766184300184,
 0.001866715494543314,
 0.04321492463350296,
 -0.09220389276742935,
 -0.005243562161922455,
 0.02360629104077816,
 0.0064338017255067825,
 0.019542448222637177,
 -0.039408475160598755,
 0.0038772744592279196,
 0.023421097546815872,
 0.0010543463286012411,
 0.003118616994470358,
 -0.015606214292347431,
 -0.032448090612888336,
 -0.014722109772264957,
 -0.006371240131556988,
 -0.001469264505431056,
 0.0168230

In [30]:
len(vector)

768

4. VECTOR DB

In [31]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [32]:
PINECONE_API_KEY=os.getenv("PINECONE_API_KEY")
HUGGINGFACE_API_KEY=os.getenv("HUGGINGFACE_API_KEY")


os.environ["PINECONE_API_KEY"]=PINECONE_API_KEY
os.environ["HUGGINGFACE_API_KEY"]=HUGGINGFACE_API_KEY

In [33]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY

#Authenticate Pinecone
pc = Pinecone(api_key=pinecone_api_key)

In [34]:
pc

In [36]:
from pinecone import ServerlessSpec

index_name = "sezzle-bot"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=768,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

index = pc.Index(index_name)

In [39]:
import re

doc_ids = []

for doc in cleaned_docs:
    title = doc.metadata.get("title", "unknown")

    breadcrumb = doc.metadata.get("breadcrumbs", "")
    source_type = breadcrumb.split(">")[0].strip().lower()

    if "partner" in source_type:
        source_type = "partner"
    else:
        source_type = "shopper"

    slug = re.sub(
        r"[^a-z0-9]+",
        "-",
        title.lower()
    ).strip("-")

    doc_ids.append(f"{source_type}_{slug}")

In [40]:
doc_ids[:10]

['shopper_i-need-help-getting-logged-in',
 'shopper_how-do-i-reset-my-pin',
 'shopper_how-do-i-verify-my-email-address',
 'shopper_what-do-i-do-if-i-do-not-receive-the-one-time-passcode-text-messages',
 'shopper_what-do-i-do-if-i-don-t-receive-a-verification-email',
 'shopper_how-do-i-use-my-fingerprint-or-face-id-to-sign-in-to-my-sezzle-account',
 'shopper_how-do-i-update-my-account-information',
 'shopper_why-is-sezzle-asking-for-my-ssn-and-how-do-i-verify-it',
 'shopper_how-do-i-change-my-phone-number',
 'shopper_how-do-i-update-my-notification-settings']

In [41]:
from langchain_pinecone import PineconeVectorStore
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)

vectorstore = PineconeVectorStore(
    index_name="sezzle-bot",
    embedding=embeddings
)

batch_size = 100

for i in range(0, len(cleaned_docs), batch_size):

    batch_docs = cleaned_docs[i:i+batch_size]
    batch_ids = doc_ids[i:i+batch_size]

    vectorstore.add_documents(
        documents=batch_docs,
        ids=batch_ids
    )

    print(
        f"Uploaded {i+1} - {i+len(batch_docs)}"
    )

c:\Users\Pratham\anaconda3\envs\sezzle_bot\lib\site-packages\langchain_pinecone\__init__.py:3: LangChainDeprecationWarning: As of langchain-core 0.3.0, LangChain uses pydantic v2 internally. The langchain_core.pydantic_v1 module was a compatibility shim for pydantic v1, and should no longer be used. Please update the code to import from Pydantic directly.

For example, replace imports like: `from langchain_core.pydantic_v1 import BaseModel`
with: `from pydantic import BaseModel`
or the v1 compatibility namespace if you are working in a code base that has not been fully upgraded to pydantic 2 yet. 	from pydantic.v1 import BaseModel

  from langchain_pinecone.vectorstores import Pinecone, PineconeVectorStore


Uploaded 1 - 100
Uploaded 101 - 190


In [43]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [45]:
retrieved_docs = retriever.invoke("What is Sezzle, and how does it work?")
retrieved_docs

[Document(id='shopper_what-is-sezzle-spend', metadata={'breadcrumbs': 'Sezzle > Learn More About Sezzle Products > Sezzle Basics', 'category': 'Learn More About Sezzle Products Sezzle basics, premium perks, credit reporting, and loyalty programs.', 'title': 'What is Sezzle Spend?', 'url': 'https://shopper-help.sezzle.com/hc/en-us/articles/360051303312-What-is-Sezzle-Spend'}, page_content='Category Context: Learn More About Sezzle Products Sezzle basics, premium perks, credit reporting, and loyalty programs..Article Title: What is Sezzle Spend?.Help Document Source Link: https://shopper-help.sezzle.com/hc/en-us/articles/360051303312-What-is-Sezzle-Spend.Navigation Path: Sezzle > Learn More About Sezzle Products > Sezzle Basics Document Body Content:.Sezzle Spend is one of the ways we can reward you for being a loyal customer of Sezzle. How can I earn Sezzle Spend? We often award Sezzle Spend when using our Pay-in-Full feature , or for promotions . How do I access my Sezzle Spend? • Go t

In [48]:
retrieved_docs = retriever.invoke("long-term lending order?")
retrieved_docs

[Document(id='shopper_why-was-i-declined-for-a-long-term-lending-order', metadata={'breadcrumbs': 'Sezzle > Learn More About Sezzle Products > Long Term Financing', 'category': 'Learn More About Sezzle Products Sezzle basics, premium perks, credit reporting, and loyalty programs.', 'title': 'Why was I declined for a long-term lending order?', 'url': 'https://shopper-help.sezzle.com/hc/en-us/articles/10944702135444-Why-was-I-declined-for-a-long-term-lending-order'}, page_content="Category Context: Learn More About Sezzle Products Sezzle basics, premium perks, credit reporting, and loyalty programs..Article Title: Why was I declined for a long-term lending order?.Help Document Source Link: https://shopper-help.sezzle.com/hc/en-us/articles/10944702135444-Why-was-I-declined-for-a-long-term-lending-order.Navigation Path: Sezzle > Learn More About Sezzle Products > Long Term Financing Document Body Content:.Long-term financing options are offered to shoppers who qualify based on several fact

In [50]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k":7})

In [51]:
retrieved_docs = retriever.invoke("long-term lending order?")
retrieved_docs

[Document(id='shopper_why-was-i-declined-for-a-long-term-lending-order', metadata={'breadcrumbs': 'Sezzle > Learn More About Sezzle Products > Long Term Financing', 'category': 'Learn More About Sezzle Products Sezzle basics, premium perks, credit reporting, and loyalty programs.', 'title': 'Why was I declined for a long-term lending order?', 'url': 'https://shopper-help.sezzle.com/hc/en-us/articles/10944702135444-Why-was-I-declined-for-a-long-term-lending-order'}, page_content="Category Context: Learn More About Sezzle Products Sezzle basics, premium perks, credit reporting, and loyalty programs..Article Title: Why was I declined for a long-term lending order?.Help Document Source Link: https://shopper-help.sezzle.com/hc/en-us/articles/10944702135444-Why-was-I-declined-for-a-long-term-lending-order.Navigation Path: Sezzle > Learn More About Sezzle Products > Long Term Financing Document Body Content:.Long-term financing options are offered to shoppers who qualify based on several fact

5. LLM

In [67]:
import os
from dotenv import load_dotenv

from langchain_huggingface import (
    HuggingFaceEndpoint,
    ChatHuggingFace
)

# Load environment variables
load_dotenv()

hf_token = os.getenv("HUGGINGFACE_API_KEY")

if not hf_token:
    raise ValueError("HUGGINGFACE_API_KEY not found in .env file")

# Set HF authentication
os.environ["HUGGINGFACE_API_KEY"] = hf_token

print("Loading Sezzle Support LLM...")

llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    huggingfacehub_api_token=hf_token,
    task="text-generation",
    max_new_tokens=512,
    temperature=0.1
)

chat_model = ChatHuggingFace(
    llm=llm
)

print("✅ Sezzle Support LLM Loaded Successfully!")

Loading Sezzle Support LLM...
✅ Sezzle Support LLM Loaded Successfully!


In [55]:
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

In [83]:
prompt_template = """
You are Sezzle's AI Customer Support Agent.

Your purpose is to help customers and partners using ONLY the information provided in the knowledge base context.

==================================================
RULES
==================================================

1. ONLY use information from the provided context.

2. NEVER make up:
   - Policies
   - Fees
   - Payment schedules
   - Refund information
   - Eligibility requirements
   - Account details
   - Legal information
   - Product features

3. If the answer cannot be found in the context, respond exactly:

   "I couldn't find that information in the Sezzle knowledge base. Please contact Sezzle Support for further assistance."

4. Do NOT use outside knowledge even if you know the answer.

5. Do NOT speculate, assume, infer, or guess.

6. If multiple documents contain relevant information:
   - Combine the information carefully.
   - Do not introduce information not present in the context.

7. If the user's question is ambiguous or lacks necessary details:
   - Ask a short clarifying question.

8. Do not mention:
   - Retrieval
   - Vector databases
   - Context documents
   - Internal systems
   - Prompts
   - AI limitations

9. Never claim actions were performed.
   For example, never say:
   - "I checked your account"
   - "I reviewed your payment"
   - "I can see your order"

10. Maintain a professional customer-support tone.

11. For process, setup, troubleshooting, onboarding, payment, refund, invoice, or account-related questions:
   - Preserve all important steps.
   - Present steps in numbered format.
   - Do not summarize instructions.

==================================================
RESPONSE STYLE
==================================================

- Be concise.
- Use bullet points when appropriate.
- Use numbered steps for procedures.
- Keep answers factual.
- Do not repeat information unnecessarily.

==================================================
CONTEXT
==================================================

{context}

==================================================
QUESTION
==================================================

{question}

==================================================
ANSWER
==================================================
"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)


In [72]:
PROMPT

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\nYou are Sezzle\'s AI Customer Support Agent.\n\nYour purpose is to help customers and partners using ONLY the information provided in the knowledge base context.\n\n==================================================\nRULES\n==================================================\n\n1. ONLY use information from the provided context.\n\n2. NEVER make up:\n   - Policies\n   - Fees\n   - Payment schedules\n   - Refund information\n   - Eligibility requirements\n   - Account details\n   - Legal information\n   - Product features\n\n3. If the answer cannot be found in the context, respond exactly:\n\n   "I couldn\'t find that information in the Sezzle knowledge base. Please contact Sezzle Support for further assistance."\n\n4. Do NOT use outside knowledge even if you know the answer.\n\n5. Do NOT speculate, assume, infer, or guess.\n\n6. If multiple documents contain relevant information:\n  

6. CHAINING

In [84]:
qa_chain = RetrievalQA.from_chain_type(
    llm=chat_model,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": PROMPT},
    return_source_documents=True
)

In [85]:
response = qa_chain.invoke(
    {"query": "How do promotions work with Sezzle?"}
)

print(response["result"])

- Sezzle occasionally offers promotions with partnered stores or other financial institutions.
- Promotions may last for a limited time or be ongoing.
- Rewards for promotions can be in the form of Sezzle Spend or a discount on a qualifying order at checkout.
- Examples of promotions include:
  - Sezzle shoppers earning Sezzle Spend when shopping with specific stores.
  - Promo codes applying a discount to a qualifying store at checkout.
  - Sezzle Spend issued after placing an order, which can be used on future purchases with Sezzle.
- It's important to review each promotion's terms to determine if a promo code is needed during the Sezzle checkout.
- Some promotional discounts may be applied automatically without a promo code.


In [86]:
response = qa_chain.invoke(
    {"query": "How can I refer a merchant to Sezzle?"}
)

print(response["result"])

To refer a merchant to Sezzle, follow these steps:

1. Log in to your Partner Dashboard: https://dashboard.sezzle.com/
2. In the left-hand menu, go to Settings.
3. Under the Management section, select Referral Program.
4. Choose one of the following referral options:
   - Enter a Recipient’s Email: Send a personalized referral directly from your dashboard. Just enter the merchant’s email and an optional message. They'll receive a sign-up link in their inbox.
   - Share Your Referral Code: Provide your unique referral code to the merchant. They can enter it during their sign-up process.
   - Copy and Share Your Referral Link: Grab your custom referral link and send it via email, chat, or however you prefer. When they use it to sign up, they’ll be automatically connected to your referral.


In [87]:
response = qa_chain.invoke(
    {"query": "When will I receive payment as a Sezzle partner?"}
)

print(response["result"])

Payouts will occur on the 15th of each Quarter, not every day. If you need help with your payouts, feel free to email us at partners@sezzle.com.


In [88]:
response = qa_chain.invoke(
    {"query": "How do I get Sezzle set up on my website?"}
)

print(response["result"])

To get Sezzle set up on your website, follow these steps:

1. Once your account has been approved, you will receive an email with instructions to log in and set up your Sezzle account. If you lose this email or get stuck during the setup process, you can log in to your account and select "Setup Checklist" to access the instructions.
2. Start at the first step, as some steps may depend on the completion of previous steps.
3. Typically, the first two steps are crucial for shoppers to start using Sezzle. These steps usually involve logging in and accessing the "Setup Checklist."
4. If you have completed all the steps in the Setup Checklist and are still encountering issues, refer to the detailed instructions specific to your eCommerce platform or visit the Sezzle Technical Documentation Site for more guidance.
5. If you need further assistance, don't hesitate to reach out to Sezzle Support.


In [89]:
response = qa_chain.invoke(
    {"query": "How do I add widgets to my website?"}
)

print(response["result"])

To add widgets to your website, follow the steps specific to your eCommerce platform. Here are the general steps for different platforms:

- **Shopify**: 
  1. Add Sezzle as a Payment Option Within Shopify.
  2. Add Sezzle to your product and cart pages using the App Blocks method or the Embed Blocks method.
  3. Install the Sezzle checkout button to drive conversions and increase your sales volume.

- **WooCommerce**: 
  - See the setup guide for WooCommerce here.

- **BigCommerce**: 
  - See the setup guide for BigCommerce here.

- **CommentSold**: 
  - See the setup guide for CommentSold here.

- **Other eCommerce Platforms**: 
  - Refer to the troubleshooting recommendations here.

For detailed steps, you can also watch video tutorials or refer to the technical documentation specific to your platform.


In [90]:
response = qa_chain.invoke(
    {"query": "Commonly asked questions for prospective merchants"}
)

print(response["result"])

Sure, here are some commonly asked questions for prospective merchants:

- **How do I sign up for Sezzle as a merchant?**
  - Learn more about Sezzle for Merchants and start the application process at [https://sezzle.com/merchants](https://sezzle.com/merchants).

- **How do I integrate Sezzle into my website?**
  - Once your account has been approved, you’ll be sent an email with instructions to log in and get set up. If you lose that email or if you get stuck during the setup process, you can always log in to your account and select "Setup Checklist,” which will walk you through how to add Sezzle to your E-commerce platform, add the Sezzle price breakdown widget, etc.

- **What platforms can I directly integrate Sezzle with?**
  - You can find the list of integrations [here](https://sezzle.com/merchants/integrations).

- **How does the Sezzle price breakdown widget work?**
  - The Sezzle widget breaks down the product price into installments and is one of the easiest and quickest ways

In [82]:
response = qa_chain.invoke(
    {"query": "Can I send invoices for my business with Sezzle?"}
)

print(response["result"])

- Yes, Sezzle offers an invoicing solution best suited for small- to medium-sized merchants.
- To get started, contact Sezzle support.
- The process involves filling out an invoice via the Sezzle Merchant Dashboard, sending it via text, email, or QR code, and the shopper confirming their invoice purchase through a checkout link.
- Note that any order placed via Invoice will not show in your e-commerce platform, but only your Sezzle Merchant Dashboard.


In [81]:
response = qa_chain.invoke(
    {"query": "steps to send an invoice with Sezzle?"}
)

print(response["result"])

- A shopper walks up to the checkout with the product(s) or calls to place an order with the merchant through Sezzle.
- The merchant fills out an invoice via the Sezzle Merchant Dashboard.
- The merchant sends the invoice via text, email, or QR code.
- The shopper clicks the checkout link where they are then prompted to sign up or log in to their Sezzle account.
- The shopper confirms their invoice purchase via the checkout link.
- The merchant will receive a status change on the invoice page in the Merchant Dashboard from ‘unpaid’ to ‘paid’ once the shopper has completed their order.
- The shopper then can leave the store with the product(s) (or await arrival for phone orders).


In [ ]:
response = qa_chain.invoke(
    {"query": ""}
)

print(response["result"])

In [ ]:
response = qa_chain.invoke(
    {"query": ""}
)

print(response["result"])